# 01 — Bias-Variance Trade-Off and Mathematical Decomposition

**Session:** 2026-11-06 · Design Validation  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

---

## 1. Mathematical Formulation of Generalization Error

In supervised learning, an algorithm observes a training set $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^N$ drawn identically and independently from an underlying data generating process:

$$y = f(x) + \epsilon, \quad \mathbb{E}[\epsilon] = 0, \quad \text{Var}(\epsilon) = \sigma^2$$

where $f(x)$ represents the true target relationship and $\epsilon$ represents irreducible stochastic noise.

When we fit a model $\hat{f}(x; \mathcal{D})$ to dataset $\mathcal{D}$, the **Expected Prediction Error (EPE)** at an unseen query point $x$ across all potential training datasets $\mathcal{D}$ is decomposed as:

$$\text{EPE}(x) = \mathbb{E}_{\mathcal{D}, \epsilon}\left[ (y - \hat{f}(x; \mathcal{D}))^2 \right]$$

### Analytical Decomposition Proof

Expanding the squared difference by adding and subtracting $\mathbb{E}_\mathcal{D}[\hat{f}(x)]$:

$$\begin{aligned}
y - \hat{f}(x) &= (f(x) + \epsilon) - \hat{f}(x) \\
&= \underbrace{(f(x) - \mathbb{E}[\hat{f}(x)])}_{-\text{Bias}(\hat{f}(x))} + \underbrace{(\mathbb{E}[\hat{f}(x)] - \hat{f}(x))}_{-\text{Estimation Deviation}} + \epsilon
\end{aligned}$$

Taking the expectation $\mathbb{E}_\mathcal{D}$ over squared terms:

$$\mathbb{E}_\mathcal{D}[(y - \hat{f}(x))^2] = \underbrace{(f(x) - \mathbb{E}[\hat{f}(x)])^2}_{\text{Bias}^2(\hat{f}(x))} + \underbrace{\mathbb{E}_\mathcal{D}[(\hat{f}(x) - \mathbb{E}[\hat{f}(x)])^2]}_{\text{Variance}(\hat{f}(x))} + \underbrace{\sigma^2}_{\text{Irreducible Noise}}$$

The cross-product terms vanish because:
1. $\mathbb{E}[\epsilon] = 0$ and $\epsilon$ is independent of the training data $\mathcal{D}$.
2. $(f(x) - \mathbb{E}[\hat{f}(x)])$ is deterministic with respect to $\mathcal{D}$.

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Reproducibility seed
np.random.seed(42)
print("Environment initialized with deterministic PRNG seed.")

## 2. Controlled Monte Carlo Simulation: Ground Truth DGP

To observe the bias-variance decomposition directly, we define a non-linear ground truth function:

$$f(x) = \cos(1.5 \pi x) + 0.1 x^2, \quad x \in [0, 1]$$

with additive Gaussian noise $\epsilon \sim \mathcal{N}(0, \sigma^2)$ where $\sigma = 0.25$.

We generate $M = 200$ independent training datasets $\mathcal{D}_1, \dots, \mathcal{D}_M$, each with $N = 40$ points, and evaluate models of varying capacity:
- **Low Capacity (Degree 1):** Linear model (High Bias, Low Variance — Underfitting).
- **Balanced Capacity (Degree 4):** Polynomial model (Low Bias, Low Variance — Optimal).
- **High Capacity (Degree 14):** High-degree polynomial (Low Bias, High Variance — Overfitting).

In [ ]:
def true_function(x: np.ndarray) -> np.ndarray:
    """True deterministic data generating function f(x)."""
    return np.cos(1.5 * np.pi * x) + 0.1 * (x**2)


def sample_dataset(n_samples: int, noise_std: float = 0.25) -> tuple[np.ndarray, np.ndarray]:
    """Draw a stochastic training dataset from the ground truth distribution."""
    x = np.sort(np.random.uniform(0.0, 1.0, size=n_samples))
    noise = np.random.normal(0.0, noise_std, size=n_samples)
    y = true_function(x) + noise
    return x, y


# Evaluate on fixed test grid
x_eval = np.linspace(0.0, 1.0, 100)
y_true = true_function(x_eval)
noise_std = 0.25
irreducible_noise = noise_std**2

print(f"Irreducible error baseline (sigma^2): {irreducible_noise:.4f}")

## 3. Empirical Bias and Variance Measurement

For each polynomial degree $d \in \{1, 4, 14\}$, we fit $\hat{f}_m(x)$ across all $M = 200$ datasets. At every query point $x_j$, we compute:

- **Mean Prediction:** $\bar{f}(x_j) = \frac{1}{M} \sum_{m=1}^M \hat{f}_m(x_j)$
- **Empirical Bias Squared:** $\text{Bias}^2(x_j) = (\bar{f}(x_j) - f(x_j))^2$
- **Empirical Variance:** $\text{Var}(x_j) = \frac{1}{M} \sum_{m=1}^M (\hat{f}_m(x_j) - \bar{f}(x_j))^2$
- **Expected Test MSE:** $\text{MSE}(x_j) = \text{Bias}^2(x_j) + \text{Var}(x_j) + \sigma^2$

In [ ]:
def fit_polynomial(x: np.ndarray, y: np.ndarray, degree: int) -> np.ndarray:
    """Fit polynomial regression coefficients using standard least squares."""
    return np.polyfit(x, y, deg=degree)


def predict_polynomial(coeffs: np.ndarray, x: np.ndarray) -> np.ndarray:
    """Evaluate polynomial predictions given coefficients and query inputs."""
    return np.polyval(coeffs, x)


num_trials = 200
n_samples = 40
degrees = [1, 4, 14]

results: dict[int, dict[str, np.ndarray | float]] = {}

for deg in degrees:
    predictions = np.zeros((num_trials, len(x_eval)))
    for trial in range(num_trials):
        x_tr, y_tr = sample_dataset(n_samples=n_samples, noise_std=noise_std)
        coeffs = fit_polynomial(x_tr, y_tr, degree=deg)
        predictions[trial, :] = predict_polynomial(coeffs, x_eval)

    # Numerical decomposition
    mean_pred = np.mean(predictions, axis=0)
    bias_sq = (mean_pred - y_true) ** 2
    variance = np.var(predictions, axis=0)
    total_mse = bias_sq + variance + irreducible_noise

    results[deg] = {
        "all_preds": predictions,
        "mean_pred": mean_pred,
        "bias_sq": float(np.mean(bias_sq)),
        "variance": float(np.mean(variance)),
        "total_mse": float(np.mean(total_mse)),
    }
    print(
        f"Degree {deg:2d} -> Bias^2: {results[deg]['bias_sq']:.4f} | "
        f"Variance: {results[deg]['variance']:.4f} | "
        f"Total Expected MSE: {results[deg]['total_mse']:.4f}"
    )

## 4. Visualizing Prediction Distributions and Model Ensembles

Under high bias (Degree 1), individual model fits are stable across training sets but deviate systematically from the true non-linear curve.
Under high variance (Degree 14), fits capture every minor fluctuation, swinging wildly at boundaries depending on which training samples were drawn.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

titles = [
    f"Degree 1 (Underfitting)\nBias^2: {results[1]['bias_sq']:.3f}, Var: {results[1]['variance']:.3f}",
    f"Degree 4 (Optimal Fit)\nBias^2: {results[4]['bias_sq']:.3f}, Var: {results[4]['variance']:.3f}",
    f"Degree 14 (Overfitting)\nBias^2: {results[14]['bias_sq']:.3f}, Var: {results[14]['variance']:.3f}",
]

for idx, deg in enumerate(degrees):
    ax = axes[idx]
    preds = results[deg]["all_preds"]
    assert isinstance(preds, np.ndarray)

    # Plot sample of fitted curves
    for i in range(25):
        ax.plot(x_eval, preds[i, :], color="steelblue", alpha=0.15, linewidth=1.0)

    # Plot mean model prediction
    mean_pred = results[deg]["mean_pred"]
    assert isinstance(mean_pred, np.ndarray)
    ax.plot(
        x_eval,
        mean_pred,
        color="crimson",
        linewidth=2.5,
        label="Expected E[f_hat]",
    )
    # Plot true target curve
    ax.plot(
        x_eval,
        y_true,
        color="black",
        linestyle="--",
        linewidth=2.0,
        label="True f(x)",
    )

    ax.set_title(titles[idx], fontsize=12, fontweight="bold")
    ax.set_xlabel("Input Feature x")
    ax.set_ylim(-1.5, 2.0)
    ax.grid(True, linestyle=":", alpha=0.6)
    if idx == 0:
        ax.set_ylabel("Response y")
        ax.legend(loc="upper right", framealpha=0.9)

plt.tight_layout()
plt.show()

## 5. Model Capacity vs. Error Curves: The Classical U-Curve

Plotting mean squared bias, variance, and total expected prediction error across degree $d \in [1, 14]$ recovers the canonical U-shaped curve that governs supervised learning.

In [ ]:
eval_degrees = list(range(1, 15))
bias_curve = []
var_curve = []
total_curve = []

for deg in eval_degrees:
    preds = np.zeros((num_trials, len(x_eval)))
    for trial in range(num_trials):
        x_tr, y_tr = sample_dataset(n_samples=n_samples, noise_std=noise_std)
        coeffs = fit_polynomial(x_tr, y_tr, degree=deg)
        preds[trial, :] = predict_polynomial(coeffs, x_eval)

    mean_p = np.mean(preds, axis=0)
    b_sq = float(np.mean((mean_p - y_true) ** 2))
    v = float(np.mean(np.var(preds, axis=0)))
    bias_curve.append(b_sq)
    var_curve.append(v)
    total_curve.append(b_sq + v + irreducible_noise)

plt.figure(figsize=(10, 6))
plt.plot(
    eval_degrees,
    bias_curve,
    marker="o",
    linewidth=2,
    color="crimson",
    label="Bias^2 (Underfitting Metric)",
)
plt.plot(
    eval_degrees,
    var_curve,
    marker="s",
    linewidth=2,
    color="royalblue",
    label="Variance (Overfitting Metric)",
)
plt.plot(
    eval_degrees,
    total_curve,
    marker="^",
    linewidth=2.5,
    color="darkgreen",
    label="Total Expected MSE",
)
plt.axhline(
    irreducible_noise,
    color="gray",
    linestyle="--",
    linewidth=1.5,
    label="Irreducible Noise (sigma^2)",
)

plt.title(
    "Empirical Bias-Variance Trade-Off Across Model Complexity",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("Polynomial Degree (Model Complexity)", fontsize=12)
plt.ylabel("Expected Error", fontsize=12)
plt.xticks(eval_degrees)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 6. Empirical Validation on Tabular Benchmark

Now we examine how empirical training error and holdout validation error diverge on realistic tabular data (`tabular_housing_validation.csv`).
Notice that while training error monotonically decreases as complexity grows, validation error reaches a minimum and subsequently explodes due to overfitting.

In [ ]:
# Load tabular housing validation dataset
data_file = Path("../data/tabular_housing_validation.csv")
records = []
with data_file.open("r", encoding="utf-8") as fp:
    reader = csv.DictReader(fp)
    for row in reader:
        records.append({k: float(v) for k, v in row.items()})

# Extract single predictive feature (med_income) and target (target_price)
x_housing = np.array([r["med_income"] for r in records])
y_housing = np.array([r["target_price"] for r in records])

# 80/20 train/validation split
indices = np.arange(len(x_housing))
np.random.shuffle(indices)
split_point = int(0.8 * len(indices))

train_idx, val_idx = indices[:split_point], indices[split_point:]
x_tr, y_tr = x_housing[train_idx], y_housing[train_idx]
x_val, y_val = x_housing[val_idx], y_housing[val_idx]

train_errors = []
val_errors = []
deg_range = list(range(1, 10))

for deg in deg_range:
    coeffs = np.polyfit(x_tr, y_tr, deg=deg)
    y_tr_pred = np.polyval(coeffs, x_tr)
    y_val_pred = np.polyval(coeffs, x_val)

    tr_mse = float(np.mean((y_tr - y_tr_pred) ** 2))
    v_mse = float(np.mean((y_val - y_val_pred) ** 2))
    train_errors.append(tr_mse)
    val_errors.append(v_mse)

best_deg = deg_range[int(np.argmin(val_errors))]
print(f"Best validation MSE attained at polynomial degree: {best_deg}")
print(
    f"Validation MSE: {min(val_errors):.4f} vs Training MSE: {train_errors[int(np.argmin(val_errors))]:.4f}"
)

## Summary: Diagnostics and Takeaways

| Failure Mode | Symptom | Bias/Variance Profile | Remediation Strategy |
| :--- | :--- | :--- | :--- |
| **High Bias (Underfitting)** | High training error, high validation error | Model capacity is too constrained | Add polynomial/interaction features, increase model complexity, reduce regularization |
| **High Variance (Overfitting)** | Low training error, high validation error | Model memorizes training noise | Increase training dataset size, add L1/L2 regularization, perform early stopping, prune features |
| **Irreducible Noise** | Validation error plateaus above zero | Noise in data collection / label error | Improve feature instrumentation, clean label errors, gather additional informative signals |